# 🚀 IBM Granite 4.0 Micro - Análise Comparativa com BERT e GPT-2

## 📋 Sobre o Modelo

**IBM Granite 4.0 Micro** é um modelo de linguagem compacto e eficiente desenvolvido pela IBM como parte da família Granite de modelos de linguagem.

### 🔗 Link para o Modelo
- **Hugging Face:** https://huggingface.co/ibm-granite/granite-4.0-micro

### 📊 Características Principais
- **Arquitetura:** Decoder-only (similar ao GPT-2)
- **Tamanho:** Versão "micro" para recursos limitados
- **Treinamento:** Multilingue incluindo português
- **Otimização:** Instrução-tuning para seguir comandos
- **Ano:** 2024 (4ª geração Granite)

### 🆚 Comparações Importantes

| Característica | BERT (2018) | GPT-2 (2019) | Granite 4.0 (2024) |
|---------------|-------------|--------------|-------------------|
| **Arquitetura** | Encoder-only | Decoder-only | Decoder-only |
| **Foco** | Compreensão | Geração | Geração + Compreensão |
| **Atenção** | Bidirecional | Causal | Causal otimizada |
| **Multilingue** | Limitado | Principalmente inglês | Sim, incluindo português |
| **Instruções** | Não | Não | Sim (instrução-tuning) |
| **Vocabulário** | ~30k tokens | ~50k tokens | ~50k tokens otimizado |

### 🎯 Aplicações Ideais
- Geração de texto em múltiplos idiomas
- Seguimento de instruções
- Tarefas gerais de processamento de linguagem
- Uso eficiente em recursos limitados

---

## 🔍 Objetivo deste Notebook

Este notebook explora o funcionamento interno do Granite 4.0 Micro, comparando-o detalhadamente com BERT e GPT-2 para entender:

1. **Tokenização:** Como cada modelo divide o texto
2. **Embeddings:** Representações vetoriais do significado
3. **Mecanismos de Atenção:** Diferenças entre bidirecional e causal
4. **Geração de Texto:** Estratégias de predição e sampling
5. **Comparações Práticas:** Vantagens e limitações de cada abordagem

---

### 💡 Por que Granite 4.0?

Granite representa a evolução natural dos modelos transformer, combinando:
- A capacidade de geração do GPT-2
- Melhor compreensão contextual
- Suporte multilingue robusto  
- Otimizações para seguir instruções
- Eficiência em recursos limitados (versão micro)

Vamos mergulhar nos detalhes técnicos e práticos! 🏊‍♂️

## Granites Usados

| Modelo | Parâmetros |
|--------|------------|
|granite-4.0-h-1b| 1b |
|granite-4.0-h-micro| 3b |
|granite-4.0-h-tiny| 7b |

## 1. Instalar dependências

In [ ]:
# Instala bibliotecas necessárias
# transformers: biblioteca principal da Hugging Face para trabalhar com modelos de linguagem
# torch: framework PyTorch para computação com tensores (GPU/CPU)
# scikit-learn: biblioteca de machine learning para métricas como similaridade
# accelerate: otimização para uso eficiente de GPU/CPU
!pip install transformers torch scikit-learn accelerate -q

2. Importar bibliotecas

In [1]:
# Importando bibliotecas principais
# AutoTokenizer: classe que carrega automaticamente o tokenizador adequado para cada modelo
# AutoModel: classe que carrega automaticamente o corpo do modelo (sem cabeça de geração)
# torch: framework principal para deep learning
# F: módulo de funções do PyTorch (softmax, etc.)
# cosine_similarity: função para calcular similaridade entre vetores
# numpy: biblioteca para computação numérica eficiente
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

3. Definir o texto

In [2]:
# Texto que será analisado
text = "O gato subiu no telhado"

In [3]:
# Carregando o tokenizer do Granite 4 Micro da IBM
# Granite 4 é um modelo LLM da IBM otimizado para entender linguagem natural
# "micro" indica uma versão compacta do modelo (menos parâmetros)
# Diferente do BERT: Granite usa arquitetura decoder-only como GPT-2
# Diferente do GPT-2: Granite foi treinado em dados multilíngues incluindo português
# "instruct" significa que é versão otimizada para seguir instruções
tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-4.0-h-1b")

# Convertendo o texto em tokens (palavras ou partes de palavras)
# Granite usa Byte-Pair Encoding (BPE) similar ao GPT-2, mas otimizado para multilinguagem
tokens = tokenizer.tokenize(text)

# Convertendo tokens em IDs numéricos
# Cada token corresponde a um ID único no vocabulário do modelo
token_ids = tokenizer.convert_tokens_to_ids(tokens)

# Exibindo resultados
print("Texto original:", text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Texto original: O gato subiu no telhado
Tokens: ['O', 'Ġg', 'ato', 'Ġsub', 'iu', 'Ġno', 'Ġtel', 'h', 'ado']
Token IDs: [46, 342, 4428, 1207, 19260, 912, 19227, 71, 2172]


5. Converter para entrada do modelo

In [4]:
# Convertendo texto para formato que o modelo entende
# return_tensors="pt" retorna tensores PyTorch
# O tokenizer adiciona automaticamente tokens especiais se necessário
# Granite usa abordagem similar ao GPT-2 (sem [CLS]/[SEP] obrigatórios como BERT)
inputs = tokenizer(text, return_tensors="pt")

# Exibindo estrutura de entrada
print(inputs)

{'input_ids': tensor([[   46,   342,  4428,  1207, 19260,   912, 19227,    71,  2172]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [6]:
# Carregando modelo Granite 4.0 Micro pré-treinado
# AutoModel carrega apenas o "corpo" do modelo (encoder/decoder)
# Diferente de AutoModelForCausalLM que inclui cabeça de geração
# Granite 4.0 Micro usa arquitetura transformer decoder-only como GPT-2
# Mas com treinamento multilingue e otimizações mais modernas
# Link: https://huggingface.co/ibm-granite/granite-4.0-h-1b
model = AutoModel.from_pretrained("ibm-granite/granite-4.0-h-1b")

# Passando o texto pelo modelo
# O modelo processa os tokens e gera embeddings (representações vetoriais)
outputs = model(**inputs)

# Pegando os embeddings (representação vetorial)
# last_hidden_state contém os embeddings para cada token
# Shape: [batch_size, sequence_length, hidden_size]
embeddings = outputs.last_hidden_state

# Exibindo dimensões
print("Shape dos embeddings:", embeddings.shape)

model.safetensors:   0%|          | 0.00/2.92G [00:00<?, ?B/s]

/home/carlosdelfino/workspace/Faculdades e Cursos/GemeosDigitais/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/466 [00:00<?, ?it/s]

[W328 15:48:58.761208483 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:00.590220591 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:02.579818482 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:04.251704727 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:06.209608099 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:08.383857620 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:10.245376538 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:12.142365509 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:14.103774552 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:49:16.109730412 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 15:4

Shape dos embeddings: torch.Size([1, 9, 1536])


7. Interpretar os embeddings

In [7]:
# Cada token agora é representado por um vetor de dimensões
# O tamanho exato depende do modelo:
# BERT base: 768 dimensões
# BERT large: 1024 dimensões  
# GPT-2 small/medium: 768/1024 dimensões
# GPT-2 large/xl: 1280/1600 dimensões
# Granite: Varia conforme o tamanho (micro geralmente menor)
print("Número de tokens:", embeddings.shape[1])
print("Dimensão de cada vetor:", embeddings.shape[2])

Número de tokens: 9
Dimensão de cada vetor: 1536


8. Similaridade entre tokens (atenção simplificada)

In [8]:
# Convertendo embeddings para numpy para cálculo de similaridade
# embeddings[0] pega o primeiro (e único) item do batch
# .detach().numpy() remove do grafo computacional e converte para numpy
emb = embeddings[0].detach().float().numpy()

# Calculando similaridade de cosseno entre todos os pares de tokens
# Isso mostra o quão semanticamente relacionados os tokens são
# Baseado em como o modelo "entende" cada palavra no contexto
similarity_matrix = cosine_similarity(emb)

# Exibindo matriz
print("Matriz de similaridade entre tokens:")
print(similarity_matrix)

Matriz de similaridade entre tokens:
[[1.         0.8810073  0.80589116 0.7735694  0.72051    0.6263846
  0.69957304 0.515968   0.77919376]
 [0.8810073  0.9999993  0.92850274 0.8540087  0.8564333  0.7718474
  0.78621274 0.62810653 0.9030498 ]
 [0.80589116 0.92850274 1.0000001  0.9141165  0.94095296 0.8738502
  0.82401377 0.6544868  0.96315384]
 [0.7735694  0.8540087  0.9141165  0.9999999  0.86594    0.8274746
  0.8260151  0.6584607  0.86575425]
 [0.72051    0.8564333  0.94095296 0.86594    1.0000001  0.9295528
  0.7982973  0.6590696  0.9706775 ]
 [0.6263846  0.7718474  0.8738502  0.8274746  0.9295528  1.0000001
  0.7883859  0.6258641  0.8977183 ]
 [0.69957304 0.78621274 0.82401377 0.8260151  0.7982973  0.7883859
  1.0000008  0.68863225 0.80280566]
 [0.515968   0.62810653 0.6544868  0.6584607  0.6590696  0.6258641
  0.68863225 1.0000012  0.6514157 ]
 [0.77919376 0.9030498  0.96315384 0.86575425 0.9706775  0.8977183
  0.80280566 0.6514157  1.0000008 ]]


9. Selecionar o último token

In [9]:
# Selecionando o vetor do último token
# Em modelos decoder-only (GPT-2, Granite), o último token carrega o contexto completo
# Diferente do BERT que usa o token [CLS] para representação da frase
# Isso ocorre porque em modelos causais, cada token acumula informação dos anteriores
last_token_vector = embeddings[0][-1]

print("Vetor do último token:")
print(last_token_vector[:10])  # mostrando apenas os primeiros valores

Vetor do último token:
tensor([  2.6719,   4.4375, -14.7500,   5.7188,  -3.5156, -11.6250,   0.8320,
         -0.8945,  -3.0469,   1.6016], dtype=torch.bfloat16,
       grad_fn=<SliceBackward0>)


10. Simular W_vocab

In [10]:
# Criando uma matriz aleatória simulando a camada de vocabulário (Language Modeling Head)
# Em modelos reais, esta matriz é aprendida durante o treinamento
# Dimensões:
# 768 -> tamanho do vetor de embedding (pode variar conforme modelo)
# 1000 -> número de palavras simuladas (vocabulários reais têm ~30k-100k tokens)
# 
# Em modelos diferentes:
# BERT: Não tem W_vocab explícito (usa camadas de classificação específicas)
# GPT-2: Tem W_vocab de [hidden_size x vocab_size] (ex: 768 x 50257)
# Granite: Similar ao GPT-2 mas com vocabulário multilingue
W_vocab = torch.randn(768, 1000)

In [13]:
# Criando uma matriz aleatória simulando a camada de vocabulário (Language Modeling Head)
# Em modelos reais, esta matriz é aprendida durante o treinamento
# Dimensões atualizadas para Granite 4.0 Micro:
# 2560 -> tamanho do vetor de embedding (observado na saída do modelo)
# 1000 -> número de palavras simuladas (vocabulários reais têm ~30k-100k tokens)
# 
# Em modelos diferentes:
# BERT: Não tem W_vocab explícito (usa camadas de classificação específicas)
# GPT-2: Tem W_vocab de [hidden_size x vocab_size] (ex: 768 x 50257)
# Granite: Similar ao GPT-2 mas com vocabulário multilingue e embedding maior
# Usando torch.bfloat16() para compatibilidade com o modelo
W_vocab = torch.randn(1536, 1000, dtype=torch.bfloat16)

In [14]:
# Multiplicação matricial: core da geração de linguagem
# Estamos comparando o vetor do contexto com todas as palavras do vocabulário
# Isso calcula "quão bem" cada palavra se encaixa no contexto atual
# 
# Diferenças entre modelos:
# BERT: Usa esta operação para classificação/tarefas específicas
# GPT-2: Usa para prever a PRÓXIMA palavra (geração)
# Granite: Similar ao GPT-2 mas com melhor compreensão contextual
logits = last_token_vector @ W_vocab

print("Shape dos logits:", logits.shape)

Shape dos logits: torch.Size([1000])


12. Converter para probabilidades (Softmax)

In [15]:
# Convertendo logits em probabilidades usando Softmax
# Softmax transforma scores brutos em distribuição de probabilidade (soma=1)
# Isso permite que o modelo "decida" qual palavra usar baseado em probabilidade
# 
# Diferenças na aplicação:
# BERT: Usa softmax para classificação (ex: sentimento positivo/negativo)
# GPT-2: Usa softmax para escolher próxima palavra na geração
# Granite: Similar ao GPT-2 mas com distribuições mais calibradas
probs = F.softmax(logits, dim=0)

# Convertendo para numpy para visualização
probs_np = probs.detach().float().numpy()

# Mostrando algumas probabilidades
print("Primeiras probabilidades:")
print(probs_np[:10])

Primeiras probabilidades:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [16]:
# Pegando os índices das maiores probabilidades
# argsort ordena do menor para o maior, então pegamos os últimos
# [::-1] inverte para ter do maior para o menor
top_k = 5
top_indices = np.argsort(probs_np)[-top_k:][::-1]

print("Top palavras simuladas (estas são de um vocabulário aleatório e não se referem ao GPT-2):")
for i, idx in enumerate(top_indices):
    print(f"Rank {i+1} - índice {idx} - probabilidade {probs_np[idx]:.4f}")

# **Comparações importantes entre modelos:**
#
# **BERT:**
# - Não foi projetado para geração de texto
# - Se tentasse gerar, teria dificuldade por ser bidirecional
# - Foco em compreensão, não em produção sequencial
#
# **GPT-2:**
# - Excelente para geração em inglês, mas limitado em português
# - Treinado principalmente em dados em inglês
# - Para obter 'telhado' como saída, precisaria de modelo treinado em português
#
# **Granite:**
# - Treinado em dados multilingues incluindo português
# - Melhor capacidade de geração em múltiplos idiomas
# - Otimizado para seguir instruções em vários idiomas

Top palavras simuladas (estas são de um vocabulário aleatório e não se referem ao GPT-2):
Rank 1 - índice 341 - probabilidade 1.0000
Rank 2 - índice 999 - probabilidade 0.0000
Rank 3 - índice 328 - probabilidade 0.0000
Rank 4 - índice 340 - probabilidade 0.0000
Rank 5 - índice 339 - probabilidade 0.0000


In [17]:
vocab_simulado = [f"palavra_{i}" for i in range(1000)]

for i, idx in enumerate(top_indices):
    print(vocab_simulado[idx], probs_np[idx])

palavra_341 1.0
palavra_999 0.0
palavra_328 0.0
palavra_340 0.0
palavra_339 0.0


## Conclusão

Resumo do processo de funcionamento das LLMs:

1. Texto → tokens (tokenização)
2. Tokens → números (input_ids)
3. Números → vetores (embeddings)
4. Vetores → relações (similaridade via atenção)
5. Último vetor representa o contexto (decoder-only) ou [CLS] (BERT)
6. Contexto é comparado com todas as palavras (W_vocab)
7. Resultado → logits (scores brutos)
8. Softmax → probabilidades (distribuição)
9. Escolha da próxima palavra (geração)

**Principais diferenças entre modelos:**

**BERT (Encoder-only):**
- Foco: Compreensão profunda
- Atenção: Bidirecional
- Representação: Token [CLS]
- Uso: Classificação, NER, QA
- Geração: Limitada

**GPT-2 (Decoder-only):**
- Foco: Geração sequencial
- Atenção: Causal (unidirecional)
- Representação: Último token
- Uso: Geração de texto, conversação
- Compreensão: Limitada

**Granite (Decoder-only moderno):**
- Foco: Geração + compreensão
- Atenção: Causal otimizada
- Representação: Último token
- Uso: Tarefas gerais, instruções
- Vantagem: Multilingue, instruções

Conhecemos agora o funcionamento básico de uma LLM e as principais diferenças arquiteturais entre os modelos mais importantes!

**Mas e se eu quiser fazer para valer?**

1. Instalar dependências

In [40]:
# Instala bibliotecas necessárias (já instaladas anteriormente, mas mantido para completude)
!pip install transformers torch -q

2. Importar bibliotecas

In [18]:
# Importando bibliotecas para geração real
# AutoModelForCausalLM: classe específica para modelos geradores de texto
# Diferente de AutoModel que só carrega o corpo do modelo
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

3. Carregar modelo e tokenizer

In [19]:
# Carregando tokenizer (converte texto ↔ números)
# AutoModelForCausalLM inclui a cabeça de geração (W_vocab + softmax)
# Diferente do AutoModel que só tem o corpo do modelo
# device_map="auto" otimiza uso de GPU/CPU automaticamente
tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-4.0-micro")

# Carregando modelo Granite 8B para geração de texto
# 8B = 8 bilhões de parâmetros (muito maior que o micro anterior)
# instruct = versão otimizada para seguir instruções
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-4.0-micro", device_map="auto")

# Colocando em modo avaliação (desliga dropout e outras camadas de treino)
model.eval()

Loading weights:   0%|          | 0/322 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


GraniteMoeHybridForCausalLM(
  (model): GraniteMoeHybridModel(
    (embed_tokens): Embedding(100352, 2560, padding_idx=100256)
    (layers): ModuleList(
      (0-39): 40 x GraniteMoeHybridDecoderLayer(
        (input_layernorm): GraniteMoeHybridRMSNorm((2560,), eps=1e-05)
        (post_attention_layernorm): GraniteMoeHybridRMSNorm((2560,), eps=1e-05)
        (shared_mlp): GraniteMoeHybridMLP(
          (activation): SiLUActivation()
          (input_linear): Linear(in_features=2560, out_features=16384, bias=False)
          (output_linear): Linear(in_features=8192, out_features=2560, bias=False)
        )
        (self_attn): GraniteMoeHybridAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=False)
          (k_proj): Linear(in_features=2560, out_features=512, bias=False)
          (v_proj): Linear(in_features=2560, out_features=512, bias=False)
          (o_proj): Linear(in_features=2560, out_features=2560, bias=False)
        )
      )
    )
    (norm): G

4. Definir texto de entrada

In [20]:
text = "O gato subiu no"

# Converter para formato do modelo
inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[   46,   342,  4428,  1207, 19260,   912]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


- Note que mudaram os valores dos tokens!

5. Rodar o modelo

In [21]:
# Passando o texto pelo modelo
with torch.no_grad():
    outputs = model(**inputs)

6. Entender a saída

In [22]:
# Pegando os logits (scores para cada palavra do vocabulário)
# outputs.logits contém os scores para CADA posição da sequência
# Shape: [batch_size, sequence_length, vocab_size]
# Para Granite 8B: vocab_size geralmente ~50k-100k tokens
logits = outputs.logits

print("Shape dos logits:", logits.shape)

# **Diferença importante em relação ao exemplo anterior:**
# No exemplo simulado: [1000] - vocabulário de 1000 palavras
# Agora: [1, 6, vocab_size] - vocabulário real de ~50k+ palavras
# E temos logits para CADA token, não só para o último

Shape dos logits: torch.Size([1, 6, 100352])


In [23]:
text_pt = "O gato subiu no"

# Converter para formato do modelo em português
# Usando o tokenizer que já foi carregado (Granite suporta português)
inputs_pt = tokenizer(text_pt, return_tensors="pt")

# Gerar continuação com sampling
output_pt = model.generate(
    **inputs_pt,
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id # Definir pad_token_id para o tokenizer
)

# Decodificar e exibir resultado
result_pt = tokenizer.decode(output_pt[0], skip_special_tokens=True)
print("Texto gerado em português:")
print(result_pt)

Texto gerado em português:
O gato subiu no topo da torre, enquanto as pessoas que iam para o trabalho continuavam a passar por ela.
3. O gato permaneceu no topo por um tempo, observando o tráfego de pedest


In [24]:
# Pegando os logits do último token
# Em modelos decoder-only, o último token contém o contexto completo
# É aqui que decidimos qual palavra gerar a seguir
last_token_logits = logits[0, -1, :]

print("Shape dos logits do último token:", last_token_logits.shape)

# **Importante:** 
# logits[0, -1, :] significa:
# 0: primeiro item do batch
# -1: último token da sequência  
# : todas as dimensões do vocabulário (~50,000 scores)
#
# **Diferença para BERT:**
# BERT usaria logits[0, 0] para o token [CLS]
# GPT-2/Granite usa o último token para geração sequencial

Shape dos logits do último token: torch.Size([100352])


8. Converter para probabilidades

In [25]:
# Convertendo logits em probabilidades usando Softmax
# Softmax: exp(x) / sum(exp(x)) para todos os x
# Transforma scores brutos em distribuição de probabilidade (soma = 1.0)
# Agora temos "quão provável" é cada palavra, não só "quão boa"
probs = F.softmax(last_token_logits, dim=0)

# **Diferença importante:**
# Logits: scores brutos (ex: 15.7, 3.2, -5.4)
# Probabilidades: distribuição normalizada (ex: 0.73, 0.15, 0.02)
#
# **Por que isso importa:**
# Logits permitem comparação direta de "confiança"
# Probabilidades permitem amostragem e tomada de decisão
#
# **Uso em diferentes modelos:**
# BERT: Usa probabilidades para classificação
# GPT-2/Granite: Usa probabilidades para escolher próxima palavra

9. Top palavras mais prováveis

In [26]:
# Pegar top 10 palavras mais prováveis
# torch.topk retorna os k maiores valores e seus índices
# Isso nos mostra o que o modelo "pensa" que são as melhores continuações
top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)

# Converter IDs para palavras para interpretação humana
for i in range(top_k):
    token_id = top_indices[i].item()
    token = tokenizer.decode([token_id])
    prob = top_probs[i].item()

    print(f"{i+1}: '{token}' → {prob:.4f}")

# **Observações importantes:**
# 1. As palavras podem ser sub-palavras (ex: " telh", "ado")
# 2. A pontuação é importante (ex: "telhado," vs "telhado")
# 3. Probabilidades geralmente baixas devido ao grande vocabulário
# 4. As melhores palavras fazem sentido contextualmente
#
# **Diferença entre modelos:**
# GPT-2: Treinado principalmente em inglês
# Granite: Treinado multilingue, melhor para português
# BERT: Não faria essa predição de próxima palavra

1: ' primeiro' → 0.1631
2: ' topo' → 0.0820
3: ' and' → 0.0364
4: ' cam' → 0.0364
5: ' ch' → 0.0283
6: ' tr' → 0.0250
7: ' tel' → 0.0250
8: ' p' → 0.0250
9: ' m' → 0.0221
10: ' c' → 0.0221


10. Veja a mágica que está revolucionando nosso mundo acontecer:

In [27]:
# Gerar continuação de texto automaticamente
# model.generate() é o método completo que faz todo o processo:
# 1. Calcula logits para próxima palavra
# 2. Aplica softmax para probabilidades  
# 3. Escolhe palavra (greedy, sampling, etc.)
# 4. Adiciona palavra ao contexto
# 5. Repete até max_length
output = model.generate(**inputs, max_length=20)

# Converter IDs para texto legível
generated_text = tokenizer.decode(output[0])

print("Texto gerado:")
print(generated_text)

# **O que aconteceu nos bastidores:**
# 1. "O gato subiu no" → logits para próxima palavra
# 2. Escolheu palavra mais provável (ex: "telhado")
# 3. "O gato subiu no telhado" → logits para próxima palavra
# 4. Escolheu palavra mais provável (ex: ",")
# 5. Repete até 20 tokens
#
# **Diferença para nossa simulação manual:**
# Nós fizemos 1 passo manualmente
# model.generate() faz todos os passos automaticamente

Texto gerado:
O gato subiu no primeiro ano, 2 kg no segundo ano e 1 kg no


11. Rodar com sampling

In [28]:
# Gerar com sampling (introduzindo criatividade e variedade)
# do_sample=True: modo de amostragem (não mais greedy)
# top_k=50: considera só as 50 palavras mais prováveis
# top_p=0.9: nucleus sampling - soma até 90% de probabilidade
# temperature=0.8: controla "criatividade" (mais alto = mais aleatório)
output = model.generate(
    **inputs,
    max_length=20,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.8
)

# Converter para texto
generated_text = tokenizer.decode(output[0])

print("Texto gerado com sampling:")
print(generated_text)

# **O que mudou?**
# Agora o modelo não pega sempre a palavra #1
# Às vezes pega a #2, #3, #5, etc.
# Introduz variedade e "criatividade"
# Menos repetições, mais surpresas
#
# **Diferença entre modelos:**
# GPT-2: Sampling funciona bem, mas limitado pelo treinamento
# Granite: Sampling mais robusto devido a melhor treinamento multilingue
# BERT: Não usa sampling (não é modelo gerador)

Texto gerado com sampling:
O gato subiu no primeiro minuto do jogo e o grito de despedida do


12. Vamos rodar com um modelo mais capaz

In [29]:

# Carregando modelo e tokenizer Granite 4
tokenizer_pt = AutoTokenizer.from_pretrained("ibm-granite/granite-4.0-micro")
model_pt = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-4.0-micro", device_map="auto")
model_pt.eval()

text_pt = "O gato subiu no"

# Converter para formato do modelo
inputs_pt = tokenizer_pt(text_pt, return_tensors="pt")

# Gerar continuação com sampling
output_pt = model_pt.generate(
    **inputs_pt,
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7,
    pad_token_id=tokenizer_pt.eos_token_id
)

# Converter para texto
generated_text_pt = tokenizer_pt.decode(output_pt[0], skip_special_tokens=True)

print("Texto gerado com Granite 4:")
print(generated_text_pt)

Loading weights:   0%|          | 0/322 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


Texto gerado com Granite 4:
O gato subiu no trilho de 6 m, e o gato é 4 metros abaixo da borda. O gato deve ter pelo menos 3 pés de distância do trilho em todas as etapas


In [43]:
# Carregando tokenizer e modelo Granite 4
tokenizer_large = AutoTokenizer.from_pretrained("ibm-granite/granite-4.0-h-tiny")

# Carregar modelo com configurações mais simples para evitar problemas
model_large = AutoModelForCausalLM.from_pretrained(
    "ibm-granite/granite-4.0-h-tiny",
    dtype=torch.float16,  # Usar float16 para economizar memória
    device_map="cpu",  # Forçar CPU para evitar problemas de meta device
    low_cpu_mem_usage=False,  # Desativar para evitar meta device
    trust_remote_code=True
)

# Colocando em modo avaliação
model_large.eval()

print("Modelo Granite 4 e tokenizer carregados!")

Loading weights:   0%|          | 0/586 [00:00<?, ?it/s]

Modelo Granite 4 e tokenizer carregados!


In [44]:
text = "O Gato subiu no"

# Converter para formato do modelo
inputs = tokenizer_large(text, return_tensors="pt")

# Gerar continuação com sampling
output = model_large.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7,
    pad_token_id=tokenizer_large.eos_token_id # Definir pad_token_id para evitar warnings
)

# Converter para texto
generated_text = tokenizer_large.decode(output[0], skip_special_tokens=True)

print("Texto gerado com Granite 4:")
print(generated_text)

[W328 17:55:31.911854782 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:55:39.266019440 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:55:42.346442607 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:55:44.500080308 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:55:46.655576141 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:55:49.294809981 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:55:52.968708951 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:55:55.798156743 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:56:01.959444637 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:56:05.983811152 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W328 17:5

Texto gerado com Granite 4:
O Gato subiu no pomar, por acaso, ou para pegar o rato?
3. Onde está o meu sapato? Não consigo encontrá-lo.
4. Por que você está se levantando tão c
